# Random Forest : exploration

Ce notebook teste la pertinence d'une forêt aléatoire avec les mêmes règles que les autres modèles : découpage commun, validation croisée stratifiée à 5 plis sur le jeu d'entraînement, PR-AUC comme métrique principale, seuil de décision fixé à 0,5 (on va opti celui du modèle finalement retenu). Le jeu de test n'est pas utilisé il sert à laisser 20 % de côté.
Chaque essai est enregistré dans MLflow pour le tracking (en tout cas normalement). 

## Importations et pré-réglages

In [ ]:
import mlflow
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from base_commune.outils import (
    RANDOM_STATE,
    charger_donnees,
    configurer_mlflow,
    decouper_donnees,
    evaluer_modele,
    exporter_runs,
    tags_reproductibilite,
)

AUTEUR = "roman"
N_ESTIMATORS = 200
CREER_LES_RUNS = False  # False : relire le notebook sans recréer de runs dans MLflow


In [16]:
df_brut = charger_donnees()
X_train, X_test, y_train, y_test = decouper_donnees(df_brut)
print(X_train.shape, X_test.shape)
# X_test et y_test ne sont JAMAIS utilisés dans ce notebook comme dit précédemment on less garde pour avoir 20% de côté à la fin

(8000, 10) (2000, 10)


## Construction du pipeline et préparation des fonctions

In [17]:
vars_num = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]
vars_cate = ["Geography", "Gender"]
vars_bin = ["HasCrCard", "IsActiveMember"]
colonnes_modele = vars_num + vars_cate + vars_bin
assert set(colonnes_modele) == set(X_train.columns), "variable oubliée ou en trop"

In [18]:
def construire_preprocesseur():
    return ColumnTransformer([
        ("cate", OneHotEncoder(handle_unknown="ignore"), vars_cate),
        ("numeriques", "passthrough", vars_num + vars_bin),], remainder="drop")


def construire_rf(n_estimators=N_ESTIMATORS, max_depth=None, min_samples_leaf=1,
                  class_weight=None):
    return Pipeline([
        ("preprocesseur", construire_preprocesseur()),
        ("rf", RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                      min_samples_leaf=min_samples_leaf,
                                      class_weight=class_weight,
                                      random_state=RANDOM_STATE, n_jobs=-1)),])

In [19]:
configurer_mlflow("churn_bancaire")

def enregistrer_run(modele, nom_run, famille, params, logger_modele=False,
                    stage="exploration", artefacts=None):
    with mlflow.start_run(run_name=nom_run):
        mlflow.set_tags({"model_family": famille, "author": AUTEUR, "stage": stage,
                         **tags_reproductibilite()})
        mlflow.log_params({**params, "features": ",".join(colonnes_modele),
                           "cv_folds": 5, "random_state": RANDOM_STATE,
                           "test_size": 0.2, "n_train": len(X_train)})
        metriques = evaluer_modele(modele, X_train, y_train)
        mlflow.log_metrics(metriques)
        if logger_modele:
            modele.fit(X_train, y_train)
            signature = infer_signature(X_train, modele.predict_proba(X_train)[:, 1])
            mlflow.sklearn.log_model(modele, name="model", signature=signature,
                                     input_example=X_train.head(3),
                                     skops_trusted_types=["sklearn.tree._tree.Tree"])
        if artefacts is not None:
            artefacts()
    return metriques

In [23]:
if CREER_LES_RUNS:
    m = enregistrer_run(DummyClassifier(strategy="prior"), "baseline_dummy_prior",
                        "baseline", {"strategy": "prior"})
    print("baseline : PR-AUC", round(m["cv_pr_auc_mean"], 3),
          "ROC-AUC", round(m["cv_roc_auc_mean"], 3))

if CREER_LES_RUNS:
    for max_depth in [8, None]:
        for min_samples_leaf in [1, 5, 20]:
            for class_weight in [None, "balanced_subsample"]:
                params = {"n_estimators": N_ESTIMATORS, "max_depth": max_depth,
                          "min_samples_leaf": min_samples_leaf,
                          "class_weight": class_weight}
                nom = f"rf_depth{max_depth}_leaf{min_samples_leaf}_cw{class_weight}"
                m = enregistrer_run(construire_rf(**params), nom, "random_forest", params)
                print(nom, "PR-AUC", round(m["cv_pr_auc_mean"], 3),
                      "recall", round(m["cv_recall_mean"], 3))

# NB je procède avec ces IF suite à une suggestion de l'ia : de base je lancais mes runs de manière "classique". J'ai voulu challenger mon notebook avec ses conseils et 
# J'ai appris qu'il valait mieux placer un paramètre True/False réglable pour éviter les pbs notamment si on fait un run all. Cette cellule modifie, sur ses conseils 
# mon code d'orgine en ce sens.

c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Us

baseline : PR-AUC 0.204 ROC-AUC 0.5
rf_depth8_leaf1_cwNone PR-AUC 0.684 recall 0.407
rf_depth8_leaf1_cwbalanced_subsample PR-AUC 0.663 recall 0.675
rf_depth8_leaf5_cwNone PR-AUC 0.681 recall 0.404
rf_depth8_leaf5_cwbalanced_subsample PR-AUC 0.669 recall 0.69
rf_depth8_leaf20_cwNone PR-AUC 0.67 recall 0.379
rf_depth8_leaf20_cwbalanced_subsample PR-AUC 0.644 recall 0.707
rf_depthNone_leaf1_cwNone PR-AUC 0.666 recall 0.453
rf_depthNone_leaf1_cwbalanced_subsample PR-AUC 0.658 recall 0.432
rf_depthNone_leaf5_cwNone PR-AUC 0.68 recall 0.447
rf_depthNone_leaf5_cwbalanced_subsample PR-AUC 0.669 recall 0.634
rf_depthNone_leaf20_cwNone PR-AUC 0.669 recall 0.395
rf_depthNone_leaf20_cwbalanced_subsample PR-AUC 0.642 recall 0.71


In [24]:
chemin = exporter_runs("churn_bancaire", AUTEUR)
print(chemin)

C:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\reports\runs_roman.csv


## COnclusion RF : 

Configuration retenue pour ce modèle : max_depth = 8, min_samples_leaf = 5, sans class_weight = PR-AUC 0,681 ; ROC-AUC 0,856, accuracy 0,858, précision 0,798, recall 0,404, F1 0,536 sans opti le seuil ce qui est encourageant et nettement mieux que le KNN 